# PHASE 1 + 2 + 3 - T0_N1 

## TO DO: 

- Check activation score waarden
- Check of microvolt of volt bandpower columns

In [31]:
import mne
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import welch
from scipy.stats import zscore

# ── Configuratie ──────────────────────────────────────────────────────────────
base_dir   = Path(r"\\vs03.herseninstituut.knaw.nl\VS03-SandC-2\raw\bnbd\Data\eeg\NSR")
output_dir = Path(r"C:\Users\zafar\Documents\bnbd_output")
output_dir.mkdir(exist_ok=True)

MAX_PARTICIPANTS = 10

SFREQ      = 256.0
WIN_SEC    = 1.0                        # 1s venster
STEP_SEC   = 0.5                        # 0.5s stap
WIN_SAMP   = int(WIN_SEC  * SFREQ)     # 256 samples
STEP_SAMP  = int(STEP_SEC * SFREQ)     # 128 samples

# Frequentiebanden
BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
}

EEG_CH = ['EEG L psg-lp', 'EEG R psg-lp']
EMG_CH = ['EEG L psg-emg', 'EEG R psg-emg']
MOV_CH = ['dX', 'dY', 'dZ']
ALL_CH = EEG_CH + EMG_CH + MOV_CH

ROLLING_SEC = 60.0   # baseline venster fase 3


# ══════════════════════════════════════════════════════════════════════════════
# FASE 1 — Load & preprocess
# ══════════════════════════════════════════════════════════════════════════════
def load_night(edf_file):
    """Laad EDF, selecteer kanalen, filter per type."""
    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose=False)
    raw.pick(ALL_CH)
    raw.load_data(verbose=False)
    raw._data = raw._data.astype(np.float64)

    # Filter EEG: 0.5–35 Hz
    raw.filter(
        l_freq=0.5, h_freq=35.0,
        picks=EEG_CH, verbose=False
    )
    # Filter EMG: 10–100 Hz (maar max sfreq/2)
    h_emg = min(100.0, SFREQ / 2 - 1)
    raw.filter(
        l_freq=10.0, h_freq=h_emg,
        picks=EMG_CH, verbose=False
    )
    # Beweging: geen filter, alleen detrenden
    raw.apply_function(
        lambda x: x - np.mean(x),
        picks=MOV_CH, verbose=False
    )
    return raw

def preprocess_signals(raw):
    """Geef data terug als dict van kanaal → signaalarray."""
    signals = {}
    for ch in ALL_CH:
        signals[ch] = raw.get_data(picks=ch)[0]
    return signals


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 2 — Feature extraction (1s vensters, 0.5s stap)
# ══════════════════════════════════════════════════════════════════════════════
def bandpower(segment, sfreq, band):
    """Gemiddeld vermogen in een frequentieband via Welch."""
    f, psd = welch(segment, fs=sfreq, nperseg=min(len(segment), WIN_SAMP))
    mask = (f >= band[0]) & (f <= band[1])
    return float(np.mean(psd[mask])) if mask.any() else 0.0


def extract_features(signals, n_total):
    """
    Schuifvenster over alle kanalen.
    Geeft een DataFrame terug met één rij per venster.
    """
    starts = np.arange(0, n_total - WIN_SAMP + 1, STEP_SAMP)
    rows   = []

    for s in starts:
        e   = s + WIN_SAMP
        t   = s / SFREQ   # tijdstempel in seconden
        row = {'time_sec': t}

        # ── EEG features ──
        for ch in EEG_CH:
            seg = signals[ch][s:e]
            tag = 'L' if 'L' in ch else 'R'

            for band_name, band_range in BANDS.items():
                row[f'eeg_{tag}_{band_name}'] = bandpower(seg, SFREQ, band_range)

            # Fast/slow ratio = (alpha+beta) / (delta+theta)
            fast = row[f'eeg_{tag}_alpha'] + row[f'eeg_{tag}_beta']
            slow = row[f'eeg_{tag}_delta'] + row[f'eeg_{tag}_theta'] + 1e-12
            row[f'eeg_{tag}_fast_slow_ratio'] = fast / slow

            # RMS
            row[f'eeg_{tag}_rms'] = float(np.sqrt(np.mean(seg ** 2)))

            # Line length (maat voor signaalcomplexiteit)
            row[f'eeg_{tag}_line_length'] = float(np.sum(np.abs(np.diff(seg))))

        # ── EMG features ──
        for ch in EMG_CH:
            seg = signals[ch][s:e]
            tag = 'L' if 'L' in ch else 'R'
            row[f'emg_{tag}_rms'] = float(np.sqrt(np.mean(seg ** 2)))

        # ── Beweging features ──
        for ch in MOV_CH:
            seg    = signals[ch][s:e]
            axis   = ch[-1].lower()   # x, y of z
            row[f'mov_{axis}_rms'] = float(np.sqrt(np.mean(seg ** 2)))

        rows.append(row)

    return pd.DataFrame(rows)


In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3 — Local baseline normalisatie
# ══════════════════════════════════════════════════════════════════════════════
def normalise_features(df):
    """
    Robuuste z-score per feature t.o.v. een rollend 60s venster.
    Rollend venster = 60s / 0.5s stap = 120 rijen.
    Geeft één activatiescore per venster (gemiddelde z-score over EEG features).
    """
    roll_rows = int(ROLLING_SEC / STEP_SEC)   # 120

    eeg_feature_cols = [
        c for c in df.columns
        if c.startswith('eeg_') and c != 'time_sec'
    ]

    df_norm = df.copy()

    for col in eeg_feature_cols:
        roll_med = df[col].rolling(roll_rows, min_periods=1, center=False).median()
        roll_mad = (
            df[col]
            .rolling(roll_rows, min_periods=1, center=False)
            .apply(lambda x: np.median(np.abs(x - np.median(x))), raw=True)
        )
        # Robuuste z-score: (x - mediaan) / (MAD + epsilon)
        df_norm[f'{col}_z'] = (df[col] - roll_med) / (roll_mad.clip(lower=1e-3) + 1e-6)
        df_norm[f'{col}_z'] = df_norm[f'{col}_z'].clip(-10, 10)

    # Activatiescore = gemiddelde z-score over alle EEG z-kolommen
    z_cols = [f'{c}_z' for c in eeg_feature_cols]
    df_norm['activation_score'] = df_norm[z_cols].mean(axis=1)

    return df_norm


In [32]:
# ══════════════════════════════════════════════════════════════════════════════
# Hoofdloop
# ══════════════════════════════════════════════════════════════════════════════
import gc

for participant_folder in sorted(base_dir.glob("bnbd_nsr_?????"))[:MAX_PARTICIPANTS]:
    pid = participant_folder.name.split("_")[-1]

    edf_file = (
        participant_folder
        / f"bnbd_nsr_{pid}_T0_N1"
        / "sleepArchitecture"
        / f"bnbd_nsr_{pid}_T0_N1_psg.edf"
    )

    if not edf_file.exists():
        print(f"[{pid}] Niet gevonden, overgeslagen.")
        continue

    # Sla over als al gedaan
    out_path = output_dir / f"features_{pid}.csv"
    if out_path.exists():
        print(f"[{pid}] Al verwerkt, overgeslagen.")
        continue

    try:
        print(f"\n[{pid}] ── Fase 1: laden & preprocessen...")
        raw     = load_night(edf_file)
        signals = preprocess_signals(raw)
        n_total = raw.n_times
        del raw
        gc.collect()

        print(f"[{pid}] ── Fase 2: feature extractie...")
        df_feat = extract_features(signals, n_total)
        del signals
        gc.collect()

        print(f"[{pid}] ── Fase 3: normalisatie...")
        df_norm = normalise_features(df_feat)
        del df_feat
        gc.collect()

        out_path = output_dir / f"features_{pid}.csv"
        df_norm.to_csv(out_path, index=False)
        print(f"[{pid}] Opgeslagen: {out_path.name}  ({len(df_norm)} vensters)")
        del df_norm
        gc.collect()

    except MemoryError:
        print(f"[{pid}] MemoryError — overgeslagen, volgende participant.")
        gc.collect()
        continue

print("\nKlaar.")

[00881] Niet gevonden, overgeslagen.
[01272] Al verwerkt, overgeslagen.
[01614] Niet gevonden, overgeslagen.
[03554] Al verwerkt, overgeslagen.
[03983] Al verwerkt, overgeslagen.
[05261] Niet gevonden, overgeslagen.

[05578] ── Fase 1: laden & preprocessen...
[05578] ── Fase 2: feature extractie...
[05578] ── Fase 3: normalisatie...
[05578] Opgeslagen: features_05578.csv  (61169 vensters)

[05830] ── Fase 1: laden & preprocessen...
[05830] ── Fase 2: feature extractie...
[05830] ── Fase 3: normalisatie...
[05830] Opgeslagen: features_05830.csv  (50929 vensters)

[07399] ── Fase 1: laden & preprocessen...
[07399] ── Fase 2: feature extractie...
[07399] ── Fase 3: normalisatie...


KeyboardInterrupt: 

**Tijdskolom**

time_sec — het tijdstempel van het begin van elk 1s venster. Dus rij 1 = seconde 0, rij 2 = seconde 0.5, rij 3 = seconde 1, etc.

EEG bandpower (L = links, R = rechts)

eeg_L_delta / eeg_R_delta — vermogen in 0.5–4 Hz. Hoog tijdens diepe slaap (N3). Bij een arousal daalt dit.
eeg_L_theta / eeg_R_theta — vermogen in 4–8 Hz. Hoog tijdens lichte slaap (N1/N2).
eeg_L_alpha / eeg_R_alpha — vermogen in 8–13 Hz. Hoog bij waken en micro-arousals. Dit is wat je wil zien pieken.
eeg_L_beta / eeg_R_beta — vermogen in 13–30 Hz. Ook hoog bij arousals en stress.

**Ratio en complexiteit**

eeg_L_fast_slow_ratio / eeg_R_fast_slow_ratio — (alpha+beta) / (delta+theta). Dit is de belangrijkste kolom voor arousal detectie: een plotselinge piek hier betekent dat het brein van langzame slaapgolven naar snelle activiteit schakelt — precies wat een micro-arousal is.
eeg_L_rms / eeg_R_rms — algemene signaalsterkte. Stijgt bij arousals maar ook bij bewegingsartefacten.
eeg_L_line_length / eeg_R_line_length — hoe veel het signaal beweegt per seconde. Hoog = complex/actief signaal.

**EMG**

emg_L_rms / emg_R_rms — spierspanning. Bij een echte micro-arousal stijgt dit vaak tegelijk met de EEG-activiteit. Handig als validatie: als alleen EMG stijgt maar EEG niet, is het waarschijnlijk een bewegingsartefact.

**Beweging**

mov_x_rms / mov_y_rms / mov_z_rms — beweging van het hoofd via de accelerometer. Als dit hoog is terwijl EEG ook hoog is, kan het een artefact zijn in plaats van een echte arousal.

**Genormaliseerde versies (_z kolommen)**

Elke kolom hierboven heeft een _z variant, bijvoorbeeld eeg_L_alpha_z. Dit is de waarde minus de lokale mediaan van de laatste 60 seconden, gedeeld door de variatie. Een waarde van 2.0 betekent dus: dit venster is 2 standaarddeviaties actiever dan de laatste minuut. Drempelwaarden voor fase 4 worden hierop gebaseerd.

**Activatiescore**

activation_score — het gemiddelde van alle EEG z-scores samen. Dit is de ene getal per venster die samenvat hoe "actief" het brein is ten opzichte van de lokale baseline. Fase 4 kijkt waar deze score boven een drempel komt om kandidaat-arousals te markeren.

Voor micro-arousal detectie wil je dus vooral kijken naar pieken in fast_slow_ratio_z en activation_score, bij voorkeur gecombineerd met een stijging in emg_rms en zonder grote uitslag in mov_rms.